[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/condensate/condensate.ipynb)

# A Rotating Bose–Einstein Condensate

This example computes the real-time evolution of a Bose–Einstein condensate stirred by a rotating elliptical harmonic trap, where quantized vortices enter from the rim and arrange into a triangular lattice. A condensate shares one quantum state across all of its atoms, so its superfluid flow follows the spatial turning of that wave rather than rigid-body rotation. Because circulation concentrates at quantized vortices, density holes around which the wave turns once, a rotating elliptical trap grips the cloud, draws vortices inward in the presence of thermal dissipation, and fills the interior at Feynman's areal density of twice the rotation rate over the circulation quantum, as in rotating superfluid helium.

In the product of the trap plane and an internal phase plane, the condensate at each vertex is modelled as a real vector in the internal `a, b` plane whose norm is the square root of the atom density and whose direction sets the local phase, with no imaginary unit. Two commuting bivectors build the rotating-frame Gross–Pitaevskii operator. The trap plane `xy` turns positions by a quarter turn via `point.commutator(xy)` into rotational drag velocity, while the internal plane `ab` turns the wave by a quarter turn via `wave * ab`. A single even element `(damping - ab) / (1 + damping ** 2)` combines real-time phase precession and thermal dissipation, and subtracting the instantaneous chemical potential preserves atom number while keeping the equilibrium phase stationary. The term ***simplex gradient*** here means the corner slope `(wave_plane & sides) / double_areas`, which pairs with the rotational drag velocity in `gradients & drag_velocities`, while summing the logarithms of normalized edge rotors `(tail * head).normalized().log()` around each triangle boundary counts every vortex as an exact integer without angle unwrapping.

In the complex scalar notation of the Gross–Pitaevskii equation ([Fetter, 2009](https://arxiv.org/abs/0801.2952)), a single imaginary unit $i$ represents both the phase precession in $(i - \gamma)\partial_t\psi = \bigl(-\tfrac12\nabla^2 + V(x,y) + g|\psi|^2 - \Omega L_z - \mu\bigr)\psi$ and the factor inside the differential angular-momentum operator $L_z = -i(x\partial_y - y\partial_x)$, and discrete plaquette winding numbers require branch-cut wrapping $\operatorname{atan2}(\operatorname{Im}(\psi_j^*\psi_k), \operatorname{Re}(\psi_j^*\psi_k))$ around each triangle.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.quantum.condensate import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext
from numga.sparse import SparseExtensor, spdiag
from numga.topology import Topology

# The trap's plane joined with the condensate's internal phase plane.
ga = Algebra("x+y+") * Algebra("a+b+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Vector+
Point = ga.gatype.antivector()                                                # Antivector-
Wave = Plane.derive.within(Algebra("a+b+"))                                   # Vector+
Turn = ga.gatype.bivector().derive.within(Algebra("a+b+"))                    # Bivector+

# Unit displacements across and up the trap plane from its centre.
horizontal = mv.x.dual()                                                      # [] Point-
vertical = mv.y.dual()                                                        # [] Point-
# The condensate's internal plane as an intrinsic patch orthogonal to spatial displacements.
wave_plane = mv.a.dual() & mv.b.dual()                                        # [] Antibivector-
# Unit inverse-mass duality map turning a spatial gradient plane into the flow it drives.
mobility = Plane.dual()                                                       # [] Point- <- Plane+

## Real-time stirring into the vortex lattice

On a concentric triangulated disk, the condensate begins at rest in a mildly elliptical harmonic trap with a Thomas–Fermi density profile and a small symmetry-breaking perturbation. Joining each opposite triangle edge with `wave_plane` forms the simplex gradients purely in the spatial directions, and pairing them across the inverse-mass duality map `Plane.dual()` gives the kinetic coupling on each triangle.

In the frame turning with the elliptical trap, multiplying the rotating-frame energy operator on the right by `(damping - ab) / (1 + damping ** 2)` couples unitary phase precession to thermal dissipation without an imaginary unit. Each real-time step advances the linear kinetic, trap, and rotational-drag terms by Crank–Nicolson while stepping contact repulsion less the instantaneous chemical potential explicitly and keeping unit atom number. The evolution and animation cell takes roughly ten seconds, split between stepping the wave and drawing the frames.

In [ ]:
topology, coordinates = scenarios.trap_disk(scenarios.n_rings)
vertices = (horizontal * coordinates[:, 0] + vertical * coordinates[:, 1]).field()   # Point-[V]
trap_potential = 0.5 * ((1 + scenarios.ellipticity) * (mv.x & vertices).squared() +
                        (1 - scenarios.ellipticity) * (mv.y & vertices).squared())   # Scalar+[V]
amplitudes, angles = scenarios.initial_cloud(coordinates)                     # [V], [V]
phase_rotors = (mv.ab * (-angles / 2)).exp()                                  # [V] Even+
raw_wave = (amplitudes * (phase_rotors >> mv.a)).field()                      # Wave+[V]

In [ ]:
def corner_assembly(topology: Topology, local: Wave) -> SparseExtensor:
    """Each triangle's `[3, 3] (Wave+ <- Wave+)[F]` corner coupling pulls back across vertex-triangle incidence to a
    global `(Wave+ <- Wave+)[V <- V]` operator."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]
    return (corners[:, None].adjugate() * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)


corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)                  # [3] Point-[F]
# Joining two edge displacements with the internal wave plane gives twice the signed triangle area.
double_areas = (sides[0] & sides[1]) & wave_plane                             # Antiscalar-[F]
areas = double_areas.antinorm() / 2                                           # Scalar+[F]
# Joining the internal wave plane with each opposite edge and dividing by twice the area gives the simplex gradient.
gradients = (wave_plane & sides) / double_areas                               # [3] Plane+[F]
dual_areas = (corners.adjugate() * (areas / 3)).sum(axis=0)                   # Scalar+[V]
initial_wave = raw_wave / (dual_areas * raw_wave.scalar_norm_squared()).sites.sum().square_root()   # Wave+[V]

In [ ]:
# Pairwise coupling of corner simplex gradients through the inverse-mass mobility over each triangle.
kinetic = (gradients[:, None] & mobility(gradients[None, :])) * (areas / 2)   # [3, 3] Scalar+[F]
# Quarter-turn velocity of each triangle centroid in the rotating frame, paired extrinsic-first with the gradients.
drag_velocities = (corners.mean(axis=0) * vertices).commutator(mv.xy)         # Point-[F]
advection = (areas / 3) * (gradients & drag_velocities)[None, :]              # [1, 3] Scalar+[F]
local_energy = kinetic * Wave - scenarios.rotation * advection * (Wave * mv.ab)   # [3, 3] (Wave+ <- Wave+)[F]
linear_energy = dual_areas * trap_potential * Wave + corner_assembly(topology, local_energy)   # (Wave+ <- Wave+)[V <- V]

# Single even phase-response element combining thermal dissipation and real-time phase precession.
phase_response = (scenarios.damping - mv.ab) / (1 + scenarios.damping ** 2)   # [] Even+
energy_rate = (dual_areas * trap_potential * (Wave * phase_response) +
               corner_assembly(topology, local_energy * phase_response))      # (Wave+ <- Wave+)[V <- V]
mass = dual_areas / scenarios.step * Wave                                     # (Wave+ <- Wave+)[V <- V]
implicit = mass + 0.5 * energy_rate                                           # (Wave+ <- Wave+)[V <- V]
explicit = mass - 0.5 * energy_rate                                           # (Wave+ <- Wave+)[V <- V]


def condensate_states(wave: Wave, n_steps: int, stride: int) -> Iterator[Wave]:
    """Yields the condensate wave (`Wave+[V]`) every `stride` real-time steps at unit atom number."""
    yield wave
    for index in range(1, n_steps + 1):
        density = wave.scalar_norm_squared()                                  # Scalar+[V]
        # Subtracting the instantaneous chemical potential preserves atom number and removes global phase precession.
        energy_wave = linear_energy(wave) + dual_areas * scenarios.repulsion * density * wave   # Wave+[V]
        chemical_potential = wave.scalar_product(energy_wave).sites.sum()     # [] Scalar+
        forcing = explicit(wave) - dual_areas * (scenarios.repulsion * density - chemical_potential) * (wave * phase_response)   # Wave+[V]
        unnormalized = implicit.solve(forcing)                                # Wave+[V]
        atom_count = (dual_areas * unnormalized.scalar_norm_squared()).sites.sum()   # [] Scalar+
        wave = unnormalized / atom_count.square_root()                        # Wave+[V]
        if index % stride == 0:
            yield wave


states = condensate_states(initial_wave, scenarios.n_steps, scenarios.stride)
evolved_states = show_states(
    states, render.wave_animation, topology, vertices, scenarios.rotation, scenarios.frame_time,
    name="condensate", duration_ms=scenarios.frame_duration_ms,
    n_frames=scenarios.n_frames, colors=scenarios.n_animation_colours,
)

Hue shows the phase of the wave, which runs once through the colour wheel around each dark vortex core, and brightness shows the atom density against the surrounding cloud, fading to black at the cores and at the rim. The white octagonal viewport represents the stationary laboratory vacuum chamber seen from the trap's rotating frame, so it turns clockwise as the elliptical trap spins counter-clockwise. Starting from rest, the cloud elongates under the rotating elliptical trap, nucleates quantized vortices along the rim, and settles into a triangular Abrikosov lattice.

## Quantized vortex windings and Feynman's rule

Along every edge of the triangulated disk, the normalized product of the endpoint wave vectors forms the rotor between them, and its bivector logarithm records the signed turn in the internal phase plane. Summing those edge turns around each triangle boundary identifies every vortex core by an exact integer winding of one full turn, and counting the vortices enclosed within concentric circles compares the settled lattice against Feynman's areal density law.

In [ ]:
def triangle_windings(topology: Topology, wave: Wave) -> Turn:
    """The logarithms of normalized edge rotors summed around each triangle boundary give its phase winding
    (`Turn+[F]`) in whole turns."""
    tail, head = topology.incidence[1, 0] * wave                              # Wave+[E], Wave+[E]
    edge_turns = (tail * head).normalized().log()                             # Turn+[E]
    return (topology.boundary[2, 1] * edge_turns).sum(axis=0) / (2 * np.pi)   # Turn+[F]


settled_wave = evolved_states[-1]                                             # Wave+[V]
centroids = corners.mean(axis=0) * vertices                                   # Point-[F]
turns = triangle_windings(topology, settled_wave).scalar_product(~mv.ab)      # Scalar+[F]
vortices = (turns > scenarios.vortex_threshold) | (turns < -scenarios.vortex_threshold)   # [F]
radii = scenarios.counting_radii                                              # [n_radii]
counts = ((centroids.antinorm().batch() <= radii[:, None]) & vortices).sum(axis=-1)   # [n_radii]
feynman_counts = scenarios.rotation * radii ** 2                              # [n_radii]
render.count_figure(radii, counts, feynman_counts);

The blue staircase shows the quantized vortices enclosed within each radius, and the orange curve shows Feynman's uniform-vorticity prediction of the rotation rate times the squared radius. The enclosed count tracks Feynman's parabola across the dense interior of the cloud before levelling off near the rim where the atom density vanishes.

In [ ]:
# checks
# Unit atom number holds on the settled condensate wave.
atom_count = (dual_areas * settled_wave.scalar_norm_squared()).sites.sum()    # [] Scalar+
assert (atom_count - 1).abs() < 1e-10

# Every vortex triangle winds once in the sense set by the rotation, and every other triangle has zero winding.
assert ((turns + 1).abs() < 1e-6)[vortices].all()
assert (turns.abs() < 1e-6)[~vortices].all()

# Across the dense interior of the cloud, the enclosed vortex count matches Feynman's areal law within 20%.
bulk = radii == 4.0                                                           # [n_radii]
assert np.allclose(counts[bulk], feynman_counts[bulk], rtol=0.2)